# Data Preparation and Training Model

## 1 - Library imports and functions

In [1]:
import os
from dotenv import load_dotenv

# Load environment variables immediately and silently
load_dotenv(override=True)

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import traceback
from matplotlib import pyplot as plt
from sklearn.metrics import r2_score
import subprocess
import json
import pickle

# Import all modules
from aq_edge.utils.logging import LoggerHandler
from aq_edge.utils.config import ConfigHandler
from aq_edge.datautils.preprocessing import prepare_data
from aq_edge.modelzoo.lstm import BaseLSTM, AttentionLSTM
from aq_edge.modelzoo.model_factory import EarlyStopping
from aq_edge.evaluation.metrics import calculate_horizon_metrics
from aq_edge.utils.visualization import (
    plot_horizon_predictions, plot_horizon_metrics, plot_loss_curves, plot_r2_curves,
    plot_forecast_vs_ground_truth, plot_single_horizon_forecast
)
from aq_edge.utils.mlflow_handler import MLflowHandler, generate_custom_run_name

[2026-10-16 14:46:11] INFO: Logging to file: D:\dly\aq_edge\artifacts\preprocessing_20261016_144611.log
[2026-10-16 14:46:11] INFO: Logging to file: D:\dly\aq_edge\artifacts\metrics_20261016_144611.log


## 2 - Start MLflow server if not running

In [2]:
logger = LoggerHandler(__name__)
logger.info("Starting air quality prediction pipeline")

# Load configuration
config = ConfigHandler()
logger.info("Configuration loaded successfully")

# Set random seeds for reproducibility
seed = config.get('training.random_seed', 42)
torch.manual_seed(seed)
np.random.seed(seed)
logger.info(f"Random seed set to: {seed}")

# MLflow integration
mlflow_enabled = config.get('mlflow.enabled', True)
custom_run_name = generate_custom_run_name()  # Generate custom run name
mlflow_handler = MLflowHandler(
    experiment_name=f"air_quality_{config.get('data.station', 'default')}",
    run_name=custom_run_name,
    enabled=mlflow_enabled
)

logger.info(f"Starting MLflow run with name: {custom_run_name}")

# Start MLflow run
if mlflow_handler.start_run(config.config):
    logger.info("MLflow run started successfully")
else:
    logger.warning("MLflow run failed to start, continuing without tracking")

logger.info("="*70)
logger.info("AIR QUALITY MODEL TRAINING AND EVALUATION")
logger.info("="*70)

[2026-10-16 14:46:26] INFO: Logging to file: D:\dly\aq_edge\artifacts\main_20261016_144626.log
INFO:__main__:Logging to file: D:\dly\aq_edge\artifacts\main_20261016_144626.log
[2026-10-16 14:46:26] INFO: Starting air quality prediction pipeline
INFO:__main__:Starting air quality prediction pipeline
[2026-10-16 14:46:26] INFO: Configuration loaded successfully
INFO:__main__:Configuration loaded successfully
[2026-10-16 14:46:26] INFO: Random seed set to: 42
INFO:__main__:Random seed set to: 42
[2026-10-16 14:46:26] INFO: Logging to file: D:\dly\aq_edge\artifacts\mlflow_handler_20261016_144626.log
INFO:aq_edge.utils.mlflow_handler:Logging to file: D:\dly\aq_edge\artifacts\mlflow_handler_20261016_144626.log
[2026-10-16 14:46:27] INFO: Starting MLflow run with name: stormy-Xochiquetzal-722
INFO:__main__:Starting MLflow run with name: stormy-Xochiquetzal-722
[2026-10-16 14:46:27] INFO: MLflow run started successfully
INFO:__main__:MLflow run started successfully
[2026-10-16 14:46:27] INFO: 

## 3 - Load data

In [3]:
station = config.get('data.station')
data_path = config.get('data.data_path').format(station=station)
drop_columns = config.get('data.drop_columns', [])

logger.info(f"Loading air quality data for station: {station}")
logger.info(f"Data path: {data_path}")

df = pd.read_parquet(data_path)
logger.info(f"Original data shape: {df.shape}")
logger.info(f"Columns: {df.columns.tolist()}")

# Drop specified columns
data = df.drop(columns=drop_columns)
logger.info(f"Dropped columns: {drop_columns}")

# Define features and target for modeling
features = config.get('data.features')
target = config.get('data.target')

logger.info(f"Features for station {station}: {features}")
logger.info(f"Target: {target}")

[2026-10-16 14:46:31] INFO: Loading air quality data for station: UTEC
INFO:__main__:Loading air quality data for station: UTEC
[2026-10-16 14:46:31] INFO: Data path: data/air/processed/UTEC_processed_features.parquet
INFO:__main__:Data path: data/air/processed/UTEC_processed_features.parquet
[2026-10-16 14:46:31] INFO: Original data shape: (5616, 52)
INFO:__main__:Original data shape: (5616, 52)
[2026-10-16 14:46:31] INFO: Columns: ['ICA', 'TEM', 'HUM', 'CO2', 'VOC', 'hour_sin1', 'hour_cos1', 'hour_sin2', 'hour_cos2', 'dow_sin', 'dow_cos', 'is_weekend', 'TEM_lag1', 'TEM_lag2', 'TEM_lag3', 'TEM_lag6', 'TEM_lag12', 'TEM_lag24', 'TEM_roll_mean6', 'TEM_roll_std6', 'TEM_roll_mean24', 'TEM_roll_std24', 'HUM_lag1', 'HUM_lag2', 'HUM_lag3', 'HUM_lag6', 'HUM_lag12', 'HUM_lag24', 'HUM_roll_mean6', 'HUM_roll_std6', 'HUM_roll_mean24', 'HUM_roll_std24', 'VOC_lag1', 'VOC_lag2', 'VOC_lag3', 'VOC_lag6', 'VOC_lag12', 'VOC_lag24', 'VOC_roll_mean6', 'VOC_roll_std6', 'VOC_roll_mean24', 'VOC_roll_std24', '

## 4 - Data Preparation

In [22]:
# -----------------------------
# 2. Prepare data
# -----------------------------
logger.info("Starting data preparation...")
preprocessing_config = config.get_section('preprocessing')

# Use the new prepare_data function
prepared_data = prepare_data(
    data=data,
    features=features,
    target=target,
    train_end_timestamp=preprocessing_config['train_end_timestamp'],
    val_end_timestamp=preprocessing_config['val_end_timestamp'],
    input_sequence_length=preprocessing_config['input_sequence_length'],
    output_sequence_length=preprocessing_config['output_sequence_length'],
    batch_size=config.get('training.batch_size')
)

train_dataloader = prepared_data['train_dataloader']
validation_dataloader = prepared_data['validation_dataloader']
test_dataloader = prepared_data['test_dataloader']

logger.info("Data preparation completed successfully!")
logger.info(f"Number of training sequences: {len(prepared_data['train_dataset'])}")
logger.info(f"Number of validation sequences: {len(prepared_data['validation_dataset'])}")
logger.info(f"Number of test sequences: {len(prepared_data['test_dataset'])}")

logger.info(f"CO2 target statistics:")
logger.info(f"Min: {data[target].min():.2f}")
logger.info(f"Max: {data[target].max():.2f}")
logger.info(f"Mean: {data[target].mean():.2f}")
logger.info(f"Std: {data[target].std():.2f}")

# MLflow: Log dataset sizes
mlflow_handler.log_metrics({
    "train_sequences": len(prepared_data['train_dataset']),
    "val_sequences": len(prepared_data['validation_dataset']),
    "test_sequences": len(prepared_data['test_dataset'])
})


[2026-10-16 14:57:05] INFO: Starting data preparation...
INFO:__main__:Starting data preparation...
[2026-10-16 14:57:05] INFO: 
[OK] scaler: standard
INFO:preprocessing:
[OK] scaler: standard
[2026-10-16 14:57:08] INFO: Data preparation completed successfully!
INFO:__main__:Data preparation completed successfully!
[2026-10-16 14:57:08] INFO: Number of training sequences: 3279
INFO:__main__:Number of training sequences: 3279
[2026-10-16 14:57:08] INFO: Number of validation sequences: 1368
INFO:__main__:Number of validation sequences: 1368
[2026-10-16 14:57:08] INFO: Number of test sequences: 900
INFO:__main__:Number of test sequences: 900
[2026-10-16 14:57:08] INFO: CO2 target statistics:
INFO:__main__:CO2 target statistics:
[2026-10-16 14:57:08] INFO: Min: 401.00
INFO:__main__:Min: 401.00
[2026-10-16 14:57:08] INFO: Max: 905.00
INFO:__main__:Max: 905.00
[2026-10-16 14:57:08] INFO: Mean: 587.14
INFO:__main__:Mean: 587.14
[2026-10-16 14:57:08] INFO: Std: 100.10
INFO:__main__:Std: 100.10

True

## 5 - Model Initialization

In [23]:
# -----------------------------
# 3. Initialize model
# -----------------------------
logger.info("Initializing model...")
use_cuda = config.get('device.use_cuda', True)
device = torch.device("cuda" if torch.cuda.is_available() and use_cuda else "cpu")
logger.info(f"Using device: {device}")

model_type = config.get('model.type', 'AttentionLSTM')
input_size = len(features)
output_size = preprocessing_config['output_sequence_length']

if model_type == 'AttentionLSTM':
    model = AttentionLSTM(input_size=input_size, output_size=output_size).to(device)
else:
    model = BaseLSTM(input_size=input_size, output_size=output_size).to(device)

logger.info(f"Model type: {model_type}")
logger.info(f"Input size: {input_size}")
logger.info(f"Output size: {output_size}")

# Initialize optimizer and criterion
training_config = config.get_section('training')
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=training_config['learning_rate'],
    weight_decay=training_config['weight_decay']
)
criterion = nn.MSELoss()

model_params = sum(p.numel() for p in model.parameters())
model_info = {
    "model_parameters": int(model_params),
    "learning_rate": float(training_config.get('learning_rate', 0)),
    "weight_decay": float(training_config.get('weight_decay', 0))
}
logger.info("Model summary: %s", json.dumps(model_info))

# MLflow: Log model info
mlflow_handler.log_metrics({
    "model_parameters": model_params,
    "input_size": input_size,
    "output_size": output_size
})

# Get git commit information (move this before skip_training logic)
try:
    current_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode('ascii').strip()
    commit_short = current_commit[:8]
except:
    current_commit = "unknown"
    commit_short = "unknown"

logger.info(f"Git commit: {commit_short}")


[2026-10-16 14:58:44] INFO: Initializing model...
INFO:__main__:Initializing model...
[2026-10-16 14:58:44] INFO: Using device: cpu
INFO:__main__:Using device: cpu
[2026-10-16 14:58:44] INFO: Model type: BaseLSTM
INFO:__main__:Model type: BaseLSTM
[2026-10-16 14:58:44] INFO: Input size: 4
INFO:__main__:Input size: 4
[2026-10-16 14:58:44] INFO: Output size: 12
INFO:__main__:Output size: 12


TypeError: LoggerHandler.info() takes 2 positional arguments but 3 were given

In [19]:
n = len(df)

In [21]:
len(df.loc[df.index > "2025-07-15"])/n

0.41203703703703703